<a href="https://colab.research.google.com/github/allengr220/cognitive-ledger/blob/main/longevity/notebooks/physiorisk_vs_phenoage_figures.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Benchmark Figures — PhysioRisk vs PhenoAgeAccel

This notebook generates publication-style figures for the aligned benchmark comparison between the locked **2-axis PhysioRisk** model and the **Levine-style no-CRP comparator**.

## Purpose

This notebook is downstream of the frozen benchmark comparison pipeline. It is used only to:

1. load frozen benchmark artifacts
2. merge any compact comparator outputs needed for plotting
3. generate Kaplan–Meier curves and related presentation figures
4. save figure files for manuscript use

## Inputs

This notebook reads frozen outputs generated upstream, including:

- benchmark-aligned scored test dataset containing:
  - `BaselineRisk`
  - `PhysioRisk_2axis`
  - `TotalRisk_2axis`
- compact comparator outputs containing:
  - `PhenoAge_noCRP_proxy`
  - `PhenoAgeAccel`

Required core columns for KM plotting:

- identifier column (`SEQN`)
- survival time (`PERMTH_INT`)
- event indicator (`MORTSTAT`)
- `PhysioRisk_2axis`
- `PhenoAgeAccel`

## Scope

This notebook is for **figure generation only**.

It does **not**:

- refit BaselineRisk
- rebuild PhysioRisk
- recompute comparator models
- alter benchmark results

All modeling decisions are frozen upstream.

## Figure plan

Primary figure(s):

1. **Kaplan–Meier curves by PhysioRisk_2axis quartile**
2. **Kaplan–Meier curves by PhenoAgeAccel quartile**

Optional figure(s):

- compact comparison figure for:
  - C-index
  - hazard ratio per 1 SD
  - age correlation

## Invariants

- All figures use the **same temporal test cohort**
- Quartiles are computed within the plotted test cohort only
- Survival endpoints are identical across figures
- No upstream model outputs are modified in this notebook

## Interpretation goal

The purpose of these figures is to visualize the contrast between:

- **PhysioRisk_2axis**, a low-age-correlation physiology score with strong mortality gradient
- **PhenoAgeAccel**, an age-adjusted comparator derived from the Levine-style no-CRP proxy

These plots are intended to support the benchmark result that PhysioRisk isolates a stronger mortality-relevant physiological signal after age adjustment.

## Output

This notebook saves final figure files (e.g. PNG or PDF) for manuscript and presentation use.

## Status

This notebook is downstream of the frozen benchmark pipeline and may be iterated for presentation only, without changing any model logic.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install lifelines

In [ ]:
# ============================================================
# Setup — journal-ready PhysioRisk figures
# ============================================================

try:
    import lifelines
except ImportError:
    !pip -q install lifelines

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.statistics import multivariate_logrank_test

# ---- Paths ----
PUB_DIR = Path("/content/drive/MyDrive/LRE/data/pub")
BENCHMARK_DIR = PUB_DIR / "benchmark_levine"
FIG_DIR = PUB_DIR / "figures_submission"
FIG_DIR.mkdir(parents=True, exist_ok=True)

TEST_SCORED_PATH = PUB_DIR / "physiorisk_2axis_v1_test_scored_from_frozen_baseline_levinenocrp.parquet"
PHENO_COMPARE_PATH = BENCHMARK_DIR / "test_phenoage_compare_outputs.parquet"
BENCHMARK_TABLE_PATH = BENCHMARK_DIR / "physiorisk_vs_phenoage_aligned_comparison.csv"

TIME_COL = "PERMTH_INT"
EVENT_COL = "MORTSTAT"
ID_COL = "SEQN"

PHYSIORISK_COL = "PhysioRisk_2axis"
PHENOACCEL_COL = "PhenoAgeAccel"

# ---- Load scored test set ----
test_df = pd.read_parquet(TEST_SCORED_PATH)

# Merge PhenoAge NCP acceleration if not already present
if PHENOACCEL_COL not in test_df.columns:
    pheno_df = pd.read_parquet(PHENO_COMPARE_PATH)
    keep_cols = [ID_COL, PHENOACCEL_COL]
    pheno_df = pheno_df[keep_cols].drop_duplicates(subset=[ID_COL])
    test_df = test_df.merge(pheno_df, on=ID_COL, how="left")

# ---- Load benchmark table ----
benchmark_table = pd.read_csv(BENCHMARK_TABLE_PATH)

# ---- Validate columns ----
required_test_cols = [TIME_COL, EVENT_COL, PHYSIORISK_COL, PHENOACCEL_COL]
missing_test = [c for c in required_test_cols if c not in test_df.columns]
if missing_test:
    raise ValueError(f"Missing required test_df columns: {missing_test}")

required_benchmark_cols = [
    "risk_score", "C_index", "HR_per_1SD",
    "HR_95CI_low", "HR_95CI_high", "corr_age"
]
missing_benchmark = [c for c in required_benchmark_cols if c not in benchmark_table.columns]
if missing_benchmark:
    raise ValueError(f"Missing required benchmark_table columns: {missing_benchmark}")

# ---- Journal-friendly plot style ----
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 9,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "axes.linewidth": 0.8,
    "lines.linewidth": 1.5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

print("Loaded test_df:", test_df.shape)
print("Loaded benchmark_table:", benchmark_table.shape)
print("Figure output directory:", FIG_DIR)


In [ ]:
# ============================================================
# Fig. 1 — Kaplan–Meier survival by quartiles
#   a PhysioRisk quartiles
#   b PhenoAge NCP acceleration quartiles
# ============================================================

def add_quartile_column(df, score_col, out_col):
    df = df.copy()
    valid = df[score_col].notna()
    # Rank stabilizes qcut if there are ties
    df.loc[valid, out_col] = pd.qcut(
        df.loc[valid, score_col].rank(method="first"),
        q=4,
        labels=["Q1 (lowest)", "Q2", "Q3", "Q4 (highest)"]
    )
    return df


def plot_km_panel(ax, df, quartile_col, panel_label):
    kmf = KaplanMeierFitter()

    line_styles = {
        "Q1 (lowest)": "-",
        "Q2": "--",
        "Q3": "-.",
        "Q4 (highest)": ":",
    }

    plot_df = df[[TIME_COL, EVENT_COL, quartile_col]].dropna().copy()

    for q in ["Q1 (lowest)", "Q2", "Q3", "Q4 (highest)"]:
        sub = plot_df[plot_df[quartile_col] == q]
        if len(sub) == 0:
            continue

        kmf.fit(
            durations=sub[TIME_COL],
            event_observed=sub[EVENT_COL],
            label=q,
        )
        kmf.plot_survival_function(
            ax=ax,
            ci_show=False,
            linestyle=line_styles[q],
        )

    result = multivariate_logrank_test(
        plot_df[TIME_COL],
        plot_df[quartile_col],
        plot_df[EVENT_COL],
    )

    ax.set_xlabel("Follow-up time (months)")
    ax.set_ylabel("Survival probability")
    ax.set_ylim(0.78, 1.01)
    ax.grid(True, linewidth=0.4, alpha=0.35)
    ax.legend(title="Risk quartile", frameon=True)

    ax.text(
        -0.12, 1.04, panel_label,
        transform=ax.transAxes,
        fontsize=11,
        fontweight="bold",
        va="top",
        ha="left",
    )

    p_label = "log-rank p < 0.001" if result.p_value < 0.001 else f"log-rank p = {result.p_value:.3g}"
    ax.text(
        0.02, 0.04,
        p_label,
        transform=ax.transAxes,
        fontsize=8,
        ha="left",
        va="bottom",
    )


fig1_df = test_df[[TIME_COL, EVENT_COL, PHYSIORISK_COL, PHENOACCEL_COL]].dropna().copy()
fig1_df = add_quartile_column(fig1_df, PHYSIORISK_COL, "PhysioRisk_quartile")
fig1_df = add_quartile_column(fig1_df, PHENOACCEL_COL, "PhenoAgeNCP_accel_quartile")

fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.4), sharey=True)

plot_km_panel(
    ax=axes[0],
    df=fig1_df,
    quartile_col="PhysioRisk_quartile",
    panel_label="a",
)

plot_km_panel(
    ax=axes[1],
    df=fig1_df,
    quartile_col="PhenoAgeNCP_accel_quartile",
    panel_label="b",
)

axes[1].set_ylabel("")
plt.tight_layout()

for ext, kwargs in {
    "png": {"dpi": 600},
    "tiff": {"dpi": 600},
    "pdf": {},
}.items():
    out = FIG_DIR / f"Fig1.{ext}"
    fig.savefig(out, bbox_inches="tight", **kwargs)
    print("Exported:", out)

plt.show()


In [ ]:
# ============================================================
# Fig. 2 — Comparative benchmark performance
#   a C-index
#   b HR per 1 SD
#   c Correlation with chronological age
# ============================================================

fig2_df = benchmark_table.copy()

# Restrict to temporal test cohort if cohort column exists
if "cohort" in fig2_df.columns:
    test_mask = fig2_df["cohort"].astype(str).str.contains("test", case=False, na=False)
    if test_mask.any():
        fig2_df = fig2_df.loc[test_mask].copy()

model_order = [
    "BaselineRisk",
    "PhenoAge_noCRP_proxy",
    "TotalRisk_2axis",
    "PhysioRisk_2axis",
    "PhenoAgeAccel",
]

label_map = {
    "BaselineRisk": "DemoRisk",
    "PhenoAge_noCRP_proxy": "PhenoAge\nNCP",
    "TotalRisk_2axis": "TotalRisk",
    "PhysioRisk_2axis": "PhysioRisk",
    "PhenoAgeAccel": "PhenoAge\nNCP accel.",
}

fig2_df = (
    fig2_df.loc[fig2_df["risk_score"].isin(model_order)]
    .assign(_order=lambda d: d["risk_score"].map({m: i for i, m in enumerate(model_order)}))
    .sort_values("_order")
    .copy()
)

if len(fig2_df) != len(model_order):
    found = set(fig2_df["risk_score"])
    missing = [m for m in model_order if m not in found]
    raise ValueError(f"Benchmark table is missing expected risk_score rows: {missing}")

labels = fig2_df["risk_score"].map(label_map).tolist()
x = np.arange(len(fig2_df))

fig, axes = plt.subplots(1, 3, figsize=(7.4, 3.1))

# Panel a: C-index
axes[0].bar(x, fig2_df["C_index"])
axes[0].set_ylabel("C-index")
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, rotation=35, ha="right")
axes[0].set_ylim(0.55, 0.90)
axes[0].grid(axis="y", linewidth=0.4, alpha=0.35)
axes[0].text(-0.18, 1.06, "a", transform=axes[0].transAxes, fontsize=11, fontweight="bold")

for i, v in enumerate(fig2_df["C_index"]):
    axes[0].text(i, v + 0.008, f"{v:.3f}", ha="center", va="bottom", fontsize=7)

# Panel b: HR per SD
hr = fig2_df["HR_per_1SD"].to_numpy()
hr_low = fig2_df["HR_95CI_low"].to_numpy()
hr_high = fig2_df["HR_95CI_high"].to_numpy()
yerr = np.vstack([hr - hr_low, hr_high - hr])

axes[1].bar(x, hr)
axes[1].errorbar(x, hr, yerr=yerr, fmt="none", capsize=3, linewidth=0.8)
axes[1].set_ylabel("HR per 1 SD")
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels, rotation=35, ha="right")
axes[1].set_ylim(0, max(hr_high) * 1.18)
axes[1].grid(axis="y", linewidth=0.4, alpha=0.35)
axes[1].text(-0.18, 1.06, "b", transform=axes[1].transAxes, fontsize=11, fontweight="bold")

for i, v in enumerate(hr):
    axes[1].text(i, v + 0.08, f"{v:.2f}", ha="center", va="bottom", fontsize=7)

# Panel c: correlation with age
axes[2].bar(x, fig2_df["corr_age"])
axes[2].set_ylabel("Correlation with age")
axes[2].set_xticks(x)
axes[2].set_xticklabels(labels, rotation=35, ha="right")
axes[2].set_ylim(0, 1.05)
axes[2].grid(axis="y", linewidth=0.4, alpha=0.35)
axes[2].text(-0.18, 1.06, "c", transform=axes[2].transAxes, fontsize=11, fontweight="bold")

for i, v in enumerate(fig2_df["corr_age"]):
    axes[2].text(i, v + 0.025, f"{v:.2f}", ha="center", va="bottom", fontsize=7)

plt.tight_layout()

for ext, kwargs in {
    "png": {"dpi": 600},
    "tiff": {"dpi": 600},
    "pdf": {},
}.items():
    out = FIG_DIR / f"Fig2.{ext}"
    fig.savefig(out, bbox_inches="tight", **kwargs)
    print("Exported:", out)

plt.show()


In [ ]:
# ============================================================
# Captions for manuscript
# ============================================================

fig1_caption = (
    "Fig. 1 Kaplan–Meier survival by quartiles of physiology-only risk scores in the temporal test cohort. "
    "a Survival curves stratified by PhysioRisk quartile. "
    "b Survival curves stratified by PhenoAge NCP acceleration quartile. "
    "PhysioRisk shows clearer separation of the highest-risk quartile, while PhenoAge NCP acceleration shows more gradual separation across quartiles. "
    "Quartiles were defined within the temporal test cohort. "
    "Differences between quartiles were statistically significant by log-rank test (p < 0.001)"
)

fig2_caption = (
    "Fig. 2 Comparative performance of mortality risk models in the temporal test cohort. "
    "a Discrimination by C-index. "
    "b Hazard ratio per one standard deviation increase with 95% confidence intervals. "
    "c Correlation with chronological age. "
    "DemoRisk and PhenoAge NCP achieved high discrimination but were strongly age-correlated. "
    "TotalRisk recovered similar discrimination while preserving explicit separation between demographic and physiological components. "
    "PhysioRisk showed lower discrimination but reduced age correlation and a larger hazard gradient than PhenoAge NCP acceleration"
)

print(fig1_caption)
print()
print(fig2_caption)
